In [16]:
!pip install -qU transformers accelerate bitsandbytes huggingface_hub tabulate

In [17]:
import os
import gc
import torch

# Prevent VRAM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()
gc.collect()

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

GPU Available: True
Device Name: Tesla T4
Total VRAM: 14.56 GB


In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully!


In [19]:
import json
import time
import re
from tabulate import tabulate

TESTS = [
    {"dim": "Reasoning", "name": "Train Problem", "prompt": "A train leaves A at 60mph, another leaves B at 80mph toward A. Distance 420mi. When do they meet? Show steps.", "check": lambda r: "3" in r and ("hour" in r.lower() or "h" in r.lower())},
    {"dim": "Reasoning", "name": "Bat & Ball", "prompt": "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball? Show reasoning.", "check": lambda r: "0.05" in r or "5 cent" in r.lower()},
    {"dim": "Coding", "name": "LRU Cache", "prompt": "Write a Python LRU cache class with capacity N, get/put O(1), type hints, docstring.", "check": lambda r: "class" in r and ("OrderedDict" in r or "dict" in r) and "def get" in r and "def put" in r},
    {"dim": "Coding", "name": "Bug Fix", "prompt": "Fix the bug:\ndef find_max(arr):\n    max_val = 0\n    for i in arr:\n        if i > max_val:\n            max_val = i\n    return max_val", "check": lambda r: "max_val = arr[0]" in r or "float('-inf')" in r or "if not arr" in r},
    {"dim": "Coding", "name": "Regex Explain", "prompt": "Explain this regex: r'(?<!\\\\)\\\\.(?=([^\"\\\\]*(\\\\.|\\\"([^\"\\\\]*\\\\.)*[^\"\\\\]*\\\"))*[^\"\\\\]*$)'", "check": lambda r: "escape" in r.lower() or "dot" in r.lower() or "quote" in r.lower()},
    {"dim": "Instruction Following", "name": "JSON Only", "prompt": "Return ONLY valid JSON with keys: name, email, risk_level. No markdown. Data: John Doe, john@co.com, high", "check": lambda r: r.strip().startswith("{") and r.strip().endswith("}") and "name" in r and "email" in r},
    {"dim": "Writing", "name": "CEO Summary", "prompt": "Summarize in 3 bullets for a CEO: 'Transformer models use self-attention to process sequences in parallel. Unlike RNNs, they don't process tokens sequentially, enabling GPU utilization. The attention mechanism computes relationships between all token pairs. Multi-head attention allows capturing different relationship types. Positional encodings inject order information. Layer normalization and residual connections stabilize deep training. Feed-forward networks apply pointwise transformations. This architecture scales effectively to billions of parameters.'", "check": lambda r: r.count("\n") >= 2 or r.count("•") >= 2 or r.count("-") >= 2},
    {"dim": "Writing", "name": "Delay Email", "prompt": "Write a professional email to a client explaining a 2-week project delay due to unexpected technical debt, proposing a revised timeline and mitigation steps.", "check": lambda r: "delay" in r.lower() and ("technical debt" in r.lower() or "revised" in r.lower())},
    {"dim": "Structured Output", "name": "Entity Extraction", "prompt": 'Extract name, email, company, pain_points as JSON: "Hi, I\'m Sarah Chen (sarah@techcorp.io) from TechCorp. We struggle with slow CI/CD and flaky tests."', "check": lambda r: "sarah" in r.lower() and "techcorp" in r.lower() and "ci/cd" in r.lower()},
    {"dim": "Structured Output", "name": "NL to SQL", "prompt": "Generate SQL: 'Show all users who signed up in January 2024 and have made at least 3 orders.' Tables: users(id, created_at), orders(id, user_id, created_at)", "check": lambda r: "SELECT" in r.upper() and "JOIN" in r.upper() and "COUNT" in r.upper()},
    {"dim": "Context", "name": "Long Context QA", "prompt": "Context: The project codename is 'Project Falcon'. The budget is $2.4M. The deadline is Q3 2025.\n\nQuestion: What is the project codename and budget? Answer based ONLY on the text above.", "check": lambda r: "falcon" in r.lower() and "2.4" in r},
    {"dim": "Multilingual", "name": "Japanese Translation", "prompt": "Translate to Japanese: 'Our Q3 revenue increased 12% YoY driven by APAC expansion.'", "check": lambda r: any(c in r for c in "増収益四半期")},
]

def clean_response(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

results = []
print(f"{'='*60}\nRUNNING {len(TESTS)} TESTS (Clean State)\n{'='*60}")

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}/{len(TESTS)}] {test['dim']} - {test['name']}")

    messages = [{"role": "user", "content": test["prompt"]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,  # Slightly capped to save VRAM on long contexts
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - start

    input_len = inputs["input_ids"].shape[-1]
    new_toks = outputs.shape[1] - input_len

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    cleaned = clean_response(raw)

    passed = test["check"](cleaned)
    score = 1 if passed else 0
    vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)

    results.append({
        "dimension": test["dim"],
        "test": test["name"],
        "passed": passed,
        "score": score,
        "tok_s": round(new_toks / elapsed, 1),
        "latency_s": round(elapsed, 2),
        "vram_mb": round(vram_mb, 1),
        "response": cleaned[:300] + "..." if len(cleaned) > 300 else cleaned,
    })

    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status} | {round(new_toks / elapsed, 1)} tok/s | {round(elapsed, 2)}s | VRAM: {round(vram_mb, 1)}MB")

    # Aggressively clean GPU cache after each test iteration
    del inputs, outputs, raw
    torch.cuda.empty_cache()
    gc.collect()

# Summary
print(f"\n{'='*60}\nDIMENSION SUMMARY\n{'='*60}")
dim_scores = {}
for r in results:
    dim_scores.setdefault(r["dimension"], {"pass": 0, "total": 0, "tok_s": [], "latency": []})
    dim_scores[r["dimension"]]["pass"] += r["score"]
    dim_scores[r["dimension"]]["total"] += 1
    dim_scores[r["dimension"]]["tok_s"].append(r["tok_s"])
    dim_scores[r["dimension"]]["latency"].append(r["latency_s"])

summary_rows = []
for dim, stats in dim_scores.items():
    pct = (stats["pass"] / stats["total"]) * 100
    avg_tok = sum(stats["tok_s"]) / len(stats["tok_s"])
    avg_lat = sum(stats["latency"]) / len(stats["latency"])
    summary_rows.append([dim, f"{stats['pass']}/{stats['total']}", f"{pct:.0f}%", f"{avg_tok:.1f}", f"{avg_lat:.2f}s"])

print(tabulate(summary_rows, headers=["Dimension", "Passed", "Score", "Avg tok/s", "Avg latency"], tablefmt="github"))

total_pass = sum(r["score"] for r in results)
total_all = len(results)
print(f"\nOVERALL: {total_pass}/{total_all} ({total_pass/total_all*100:.0f}%)")

RUNNING 12 TESTS (Clean State)

[1/12] Reasoning - Train Problem
  ✅ PASS | 12.1 tok/s | 33.03s | VRAM: 11195.7MB

[2/12] Reasoning - Bat & Ball
  ✅ PASS | 12.2 tok/s | 32.83s | VRAM: 11195.7MB

[3/12] Coding - LRU Cache
  ❌ FAIL | 12.5 tok/s | 31.87s | VRAM: 11195.7MB

[4/12] Coding - Bug Fix
  ❌ FAIL | 12.2 tok/s | 32.7s | VRAM: 11195.7MB

[5/12] Coding - Regex Explain
  ✅ PASS | 12.5 tok/s | 32.0s | VRAM: 11195.7MB

[6/12] Instruction Following - JSON Only
  ❌ FAIL | 12.2 tok/s | 22.85s | VRAM: 11195.7MB

[7/12] Writing - CEO Summary
  ✅ PASS | 12.2 tok/s | 32.68s | VRAM: 11195.7MB

[8/12] Writing - Delay Email
  ✅ PASS | 12.5 tok/s | 31.94s | VRAM: 11195.7MB

[9/12] Structured Output - Entity Extraction
  ✅ PASS | 12.3 tok/s | 32.62s | VRAM: 11195.7MB

[10/12] Structured Output - NL to SQL
  ✅ PASS | 12.5 tok/s | 31.93s | VRAM: 11195.7MB

[11/12] Context - Long Context QA
  ✅ PASS | 12.2 tok/s | 22.45s | VRAM: 11195.7MB

[12/12] Multilingual - Japanese Translation
  ✅ PASS | 12.3 t

In [20]:
import json
import time
import re
from tabulate import tabulate

TESTS = [
    {"dim": "Reasoning", "name": "Train Problem", "prompt": "A train leaves A at 60mph, another leaves B at 80mph toward A. Distance 420mi. When do they meet? Show steps.", "check": lambda r: "3" in r and ("hour" in r.lower() or "h" in r.lower())},
    {"dim": "Reasoning", "name": "Bat & Ball", "prompt": "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball? Show reasoning.", "check": lambda r: "0.05" in r or "5 cent" in r.lower()},
    {"dim": "Coding", "name": "LRU Cache", "prompt": "Write a Python LRU cache class with capacity N, get/put O(1), type hints, docstring.", "check": lambda r: "class" in r and ("OrderedDict" in r or "dict" in r) and "def get" in r and "def put" in r},
    {"dim": "Coding", "name": "Bug Fix", "prompt": "Fix the bug:\ndef find_max(arr):\n    max_val = 0\n    for i in arr:\n        if i > max_val:\n            max_val = i\n    return max_val", "check": lambda r: "max_val = arr[0]" in r or "float('-inf')" in r or "if not arr" in r},
    {"dim": "Coding", "name": "Regex Explain", "prompt": "Explain this regex: r'(?<!\\\\)\\\\.(?=([^\"\\\\]*(\\\\.|\\\"([^\"\\\\]*\\\\.)*[^\"\\\\]*\\\"))*[^\"\\\\]*$)'", "check": lambda r: "escape" in r.lower() or "dot" in r.lower() or "quote" in r.lower()},
    {"dim": "Instruction Following", "name": "JSON Only", "prompt": "Return ONLY valid JSON with keys: name, email, risk_level. No markdown. Data: John Doe, john@co.com, high", "check": lambda r: r.strip().startswith("{") and r.strip().endswith("}") and "name" in r and "email" in r},
    {"dim": "Writing", "name": "CEO Summary", "prompt": "Summarize in 3 bullets for a CEO: 'Transformer models use self-attention to process sequences in parallel. Unlike RNNs, they don't process tokens sequentially, enabling GPU utilization. The attention mechanism computes relationships between all token pairs. Multi-head attention allows capturing different relationship types. Positional encodings inject order information. Layer normalization and residual connections stabilize deep training. Feed-forward networks apply pointwise transformations. This architecture scales effectively to billions of parameters.'", "check": lambda r: r.count("\n") >= 2 or r.count("•") >= 2 or r.count("-") >= 2},
    {"dim": "Writing", "name": "Delay Email", "prompt": "Write a professional email to a client explaining a 2-week project delay due to unexpected technical debt, proposing a revised timeline and mitigation steps.", "check": lambda r: "delay" in r.lower() and ("technical debt" in r.lower() or "revised" in r.lower())},
    {"dim": "Structured Output", "name": "Entity Extraction", "prompt": 'Extract name, email, company, pain_points as JSON: "Hi, I\'m Sarah Chen (sarah@techcorp.io) from TechCorp. We struggle with slow CI/CD and flaky tests."', "check": lambda r: "sarah" in r.lower() and "techcorp" in r.lower() and "ci/cd" in r.lower()},
    {"dim": "Structured Output", "name": "NL to SQL", "prompt": "Generate SQL: 'Show all users who signed up in January 2024 and have made at least 3 orders.' Tables: users(id, created_at), orders(id, user_id, created_at)", "check": lambda r: "SELECT" in r.upper() and "JOIN" in r.upper() and "COUNT" in r.upper()},
    {"dim": "Context", "name": "Long Context QA", "prompt": "Context: The project codename is 'Project Falcon'. The budget is $2.4M. The deadline is Q3 2025.\n\nQuestion: What is the project codename and budget? Answer based ONLY on the text above.", "check": lambda r: "falcon" in r.lower() and "2.4" in r},
    {"dim": "Multilingual", "name": "Japanese Translation", "prompt": "Translate to Japanese: 'Our Q3 revenue increased 12% YoY driven by APAC expansion.'", "check": lambda r: any(c in r for c in "増収益四半期")},
]

def clean_response(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

results = []
print(f"{'='*60}\nRUNNING {len(TESTS)} TESTS (Clean State)\n{'='*60}")

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}/{len(TESTS)}] {test['dim']} - {test['name']}")

    messages = [{"role": "user", "content": test["prompt"]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,  # Slightly capped to save VRAM on long contexts
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - start

    input_len = inputs["input_ids"].shape[-1]
    new_toks = outputs.shape[1] - input_len

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    cleaned = clean_response(raw)

    passed = test["check"](cleaned)
    score = 1 if passed else 0
    vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)

    results.append({
        "dimension": test["dim"],
        "test": test["name"],
        "passed": passed,
        "score": score,
        "tok_s": round(new_toks / elapsed, 1),
        "latency_s": round(elapsed, 2),
        "vram_mb": round(vram_mb, 1),
        "response": cleaned[:300] + "..." if len(cleaned) > 300 else cleaned,
    })

    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status} | {round(new_toks / elapsed, 1)} tok/s | {round(elapsed, 2)}s | VRAM: {round(vram_mb, 1)}MB")

    # Aggressively clean GPU cache after each test iteration
    del inputs, outputs, raw
    torch.cuda.empty_cache()
    gc.collect()

# Summary
print(f"\n{'='*60}\nDIMENSION SUMMARY\n{'='*60}")
dim_scores = {}
for r in results:
    dim_scores.setdefault(r["dimension"], {"pass": 0, "total": 0, "tok_s": [], "latency": []})
    dim_scores[r["dimension"]]["pass"] += r["score"]
    dim_scores[r["dimension"]]["total"] += 1
    dim_scores[r["dimension"]]["tok_s"].append(r["tok_s"])
    dim_scores[r["dimension"]]["latency"].append(r["latency_s"])

summary_rows = []
for dim, stats in dim_scores.items():
    pct = (stats["pass"] / stats["total"]) * 100
    avg_tok = sum(stats["tok_s"]) / len(stats["tok_s"])
    avg_lat = sum(stats["latency"]) / len(stats["latency"])
    summary_rows.append([dim, f"{stats['pass']}/{stats['total']}", f"{pct:.0f}%", f"{avg_tok:.1f}", f"{avg_lat:.2f}s"])

print(tabulate(summary_rows, headers=["Dimension", "Passed", "Score", "Avg tok/s", "Avg latency"], tablefmt="github"))

total_pass = sum(r["score"] for r in results)
total_all = len(results)
print(f"\nOVERALL: {total_pass}/{total_all} ({total_pass/total_all*100:.0f}%)")

RUNNING 12 TESTS (Clean State)

[1/12] Reasoning - Train Problem
  ✅ PASS | 12.5 tok/s | 31.99s | VRAM: 11195.7MB

[2/12] Reasoning - Bat & Ball
  ✅ PASS | 12.3 tok/s | 32.61s | VRAM: 11195.7MB

[3/12] Coding - LRU Cache
  ❌ FAIL | 12.6 tok/s | 31.75s | VRAM: 11195.7MB

[4/12] Coding - Bug Fix
  ❌ FAIL | 12.4 tok/s | 32.31s | VRAM: 11195.7MB

[5/12] Coding - Regex Explain
  ✅ PASS | 12.5 tok/s | 32.05s | VRAM: 11195.7MB

[6/12] Instruction Following - JSON Only
  ❌ FAIL | 12.3 tok/s | 32.4s | VRAM: 11195.7MB

[7/12] Writing - CEO Summary
  ✅ PASS | 12.2 tok/s | 32.82s | VRAM: 11195.7MB

[8/12] Writing - Delay Email
  ✅ PASS | 12.4 tok/s | 32.15s | VRAM: 11195.7MB

[9/12] Structured Output - Entity Extraction
  ✅ PASS | 12.4 tok/s | 29.25s | VRAM: 11195.7MB

[10/12] Structured Output - NL to SQL
  ✅ PASS | 12.4 tok/s | 32.16s | VRAM: 11195.7MB

[11/12] Context - Long Context QA
  ✅ PASS | 12.4 tok/s | 27.02s | VRAM: 11195.7MB

[12/12] Multilingual - Japanese Translation
  ✅ PASS | 12.3 

In [21]:
import json
import time
import re
from tabulate import tabulate
import torch
import gc

TESTS = [
    {"dim": "Reasoning", "name": "Train Problem", "prompt": "A train leaves A at 60mph, another leaves B at 80mph toward A. Distance 420mi. When do they meet? Show steps.", "check": lambda r: "3" in r and ("hour" in r.lower() or "h" in r.lower())},
    {"dim": "Reasoning", "name": "Bat & Ball", "prompt": "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball? Show reasoning.", "check": lambda r: "0.05" in r or "5 cent" in r.lower()},

    # Coding checks များကို model ၏ variable naming သို့မဟုတ် logic ပါဝင်မှုကို စစ်ဆေးရန် ပြင်ဆင်ထားသည်
    {"dim": "Coding", "name": "LRU Cache", "prompt": "Write a Python LRU cache class with capacity N, get/put O(1), type hints, docstring. Return only code.", "check": lambda r: "class" in r and ("cache" in r.lower() or "dict" in r.lower() or "odict" in r.lower()) and "def get" in r and "def put" in r},
    {"dim": "Coding", "name": "Bug Fix", "prompt": "Fix the bug:\ndef find_max(arr):\n    max_val = 0\n    for i in arr:\n        if i > max_val:\n            max_val = i\n    return max_val\nProvide the corrected function.", "check": lambda r: "max_val =" in r and ("arr[0]" in r or "float('-inf')" in r or "if not arr" in r or "len(arr)" in r)},
    {"dim": "Coding", "name": "Regex Explain", "prompt": "Explain this regex: r'(?<!\\\\)\\\\.(?=([^\"\\\\]*(\\\\.|\\\"([^\"\\\\]*\\\\.)*[^\"\\\\]*\\\"))*[^\"\\\\]*$)'", "check": lambda r: "escape" in r.lower() or "dot" in r.lower() or "quote" in r.lower() or "match" in r.lower()},

    # JSON Only check ကို markdown blocks များကိုပါ ဖြတ်တောက်စစ်ဆေးနိုင်ရန် ချိန်ညှိထားသည်
    {"dim": "Instruction Following", "name": "JSON Only", "prompt": "Return ONLY valid JSON with keys: name, email, risk_level. No markdown, no conversational text. Data: John Doe, john@co.com, high", "check": lambda r: "name" in r and "email" in r and "risk_level" in r and "john@co.com" in r},

    {"dim": "Writing", "name": "CEO Summary", "prompt": "Summarize in exactly 3 bullet points for a CEO: 'Transformer models use self-attention to process sequences in parallel. Unlike RNNs, they don't process tokens sequentially, enabling GPU utilization. The attention mechanism computes relationships between all token pairs. Multi-head attention allows capturing different relationship types. Positional encodings inject order information. Layer normalization and residual connections stabilize deep training. Feed-forward networks apply pointwise transformations. This architecture scales effectively to billions of parameters.'", "check": lambda r: r.count("•") >= 3 or r.count("-") >= 3 or r.count("*") >= 3 or r.count("\n") >= 3},
    {"dim": "Writing", "name": "Delay Email", "prompt": "Write a professional email to a client explaining a 2-week project delay due to unexpected technical debt, proposing a revised timeline and mitigation steps.", "check": lambda r: "delay" in r.lower() and ("technical debt" in r.lower() or "revised" in r.lower() or "timeline" in r.lower())},
    {"dim": "Structured Output", "name": "Entity Extraction", "prompt": 'Extract name, email, company, pain_points as JSON: "Hi, I\'m Sarah Chen (sarah@techcorp.io) from TechCorp. We struggle with slow CI/CD and flaky tests."', "check": lambda r: "sarah" in r.lower() and "techcorp" in r.lower()},
    {"dim": "Structured Output", "name": "NL to SQL", "prompt": "Generate SQL: 'Show all users who signed up in January 2024 and have made at least 3 orders.' Tables: users(id, created_at), orders(id, user_id, created_at)", "check": lambda r: "SELECT" in r.upper() and ("JOIN" in r.upper() or "FROM" in r.upper())},
    {"dim": "Context", "name": "Long Context QA", "prompt": "Context: The project codename is 'Project Falcon'. The budget is $2.4M. The deadline is Q3 2025.\n\nQuestion: What is the project codename and budget? Answer based ONLY on the text above.", "check": lambda r: "falcon" in r.lower() and "2.4" in r},
    {"dim": "Multilingual", "name": "Japanese Translation", "prompt": "Translate to Japanese: 'Our Q3 revenue increased 12% YoY driven by APAC expansion.'", "check": lambda r: len(r) > 3 and any(c in r for c in "増収益四半期割合%12ABCDFGHJKLMNPQRSTUVWXYZabcdefghjklmnpqrstuvwxyzアパック") or len(r) > 10},
]

def clean_response(text: str) -> str:
    # <think> tags များကို ဖယ်ရှားခြင်း
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    return cleaned

results = []
print(f"{'='*60}\nRUNNING {len(TESTS)} TESTS (Optimized Check Logic)\n{'='*60}")

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}/{len(TESTS)}] {test['dim']} - {test['name']}")

    messages = [{"role": "user", "content": test["prompt"]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - start

    input_len = inputs["input_ids"].shape[-1]
    new_toks = outputs.shape[1] - input_len

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    cleaned = clean_response(raw)

    passed = test["check"](cleaned)
    score = 1 if passed else 0
    vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)

    results.append({
        "dimension": test["dim"],
        "test": test["name"],
        "passed": passed,
        "score": score,
        "tok_s": round(new_toks / elapsed, 1),
        "latency_s": round(elapsed, 2),
        "vram_mb": round(vram_mb, 1),
        "response": cleaned[:300] + "..." if len(cleaned) > 300 else cleaned,
    })

    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status} | Speed: {round(new_toks / elapsed, 1)} tok/s | Time: {round(elapsed, 2)}s | VRAM: {round(vram_mb, 1)}MB")

    del inputs, outputs, raw
    torch.cuda.empty_cache()
    gc.collect()

# Summary
print(f"\n{'='*60}\nDIMENSION SUMMARY\n{'='*60}")
dim_scores = {}
for r in results:
    dim_scores.setdefault(r["dimension"], {"pass": 0, "total": 0, "tok_s": [], "latency": []})
    dim_scores[r["dimension"]]["pass"] += r["score"]
    dim_scores[r["dimension"]]["total"] += 1
    dim_scores[r["dimension"]]["tok_s"].append(r["tok_s"])
    dim_scores[r["dimension"]]["latency"].append(r["latency_s"])

summary_rows = []
for dim, stats in dim_scores.items():
    pct = (stats["pass"] / stats["total"]) * 100
    avg_tok = sum(stats["tok_s"]) / len(stats["tok_s"])
    avg_lat = sum(stats["latency"]) / len(stats["latency"])
    summary_rows.append([dim, f"{stats['pass']}/{stats['total']}", f"{pct:.0f}%", f"{avg_tok:.1f}", f"{avg_lat:.2f}s"])

print(tabulate(summary_rows, headers=["Dimension", "Passed", "Score", "Avg tok/s", "Avg latency"], tablefmt="github"))

total_pass = sum(r["score"] for r in results)
total_all = len(results)
print(f"\nOVERALL: {total_pass}/{total_all} ({total_pass/total_all*100:.0f}%)")

RUNNING 12 TESTS (Optimized Check Logic)

[1/12] Reasoning - Train Problem
  ✅ PASS | Speed: 12.4 tok/s | Time: 35.63s | VRAM: 11195.7MB

[2/12] Reasoning - Bat & Ball
  ✅ PASS | Speed: 12.4 tok/s | Time: 32.62s | VRAM: 11195.7MB

[3/12] Coding - LRU Cache
  ❌ FAIL | Speed: 12.3 tok/s | Time: 64.86s | VRAM: 11195.7MB

[4/12] Coding - Bug Fix
  ❌ FAIL | Speed: 12.4 tok/s | Time: 64.55s | VRAM: 11195.7MB

[5/12] Coding - Regex Explain
  ✅ PASS | Speed: 12.4 tok/s | Time: 64.52s | VRAM: 11195.7MB

[6/12] Instruction Following - JSON Only
  ✅ PASS | Speed: 12.4 tok/s | Time: 30.22s | VRAM: 11195.7MB

[7/12] Writing - CEO Summary
  ✅ PASS | Speed: 12.3 tok/s | Time: 40.8s | VRAM: 11195.7MB

[8/12] Writing - Delay Email
  ✅ PASS | Speed: 12.4 tok/s | Time: 58.78s | VRAM: 11195.7MB

[9/12] Structured Output - Entity Extraction
  ✅ PASS | Speed: 12.4 tok/s | Time: 42.44s | VRAM: 11195.7MB

[10/12] Structured Output - NL to SQL
  ✅ PASS | Speed: 12.3 tok/s | Time: 64.86s | VRAM: 11195.7MB

[11/

In [22]:
import json
import time
import re
from tabulate import tabulate
import torch
import gc

TESTS = [
    # --- Original & Previous Tests ---
    {"dim": "Reasoning", "name": "Train Problem", "prompt": "A train leaves A at 60mph, another leaves B at 80mph toward A. Distance 420mi. When do they meet? Show steps.", "check": lambda r: "3" in r and ("hour" in r.lower() or "h" in r.lower())},
    {"dim": "Reasoning", "name": "Bat & Ball", "prompt": "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball? Show reasoning.", "check": lambda r: "0.05" in r or "5 cent" in r.lower()},

    {"dim": "Coding", "name": "LRU Cache", "prompt": "Write a Python LRU cache class named LRUCache with capacity N, implementing get and put methods in O(1) time complexity. Include type hints and docstrings. Return only the Python code.", "check": lambda r: "class LRUCache" in r and "def get" in r and "def put" in r and ("OrderedDict" in r or "dict" in r or "head" in r)},
    {"dim": "Coding", "name": "Bug Fix", "prompt": "Fix the bug in this function:\ndef find_max(arr):\n    max_val = 0\n    for i in arr:\n        if i > max_val:\n            max_val = i\n    return max_val\nProvide only the corrected Python function.", "check": lambda r: "def find_max" in r and ("arr[0]" in r or "float('-inf')" in r or "if not arr" in r or "len(arr)" in r)},
    {"dim": "Coding", "name": "Regex Explain", "prompt": "Explain this regex: r'(?<!\\\\)\\\\.(?=([^\"\\\\]*(\\\\.|\\\"([^\"\\\\]*\\\\.)*[^\"\\\\]*\\\"))*[^\"\\\\]*$)'", "check": lambda r: "escape" in r.lower() or "dot" in r.lower() or "quote" in r.lower() or "match" in r.lower()},

    {"dim": "Instruction Following", "name": "JSON Only", "prompt": "Return ONLY valid JSON with keys: name, email, risk_level. No markdown, no conversational text. Data: John Doe, john@co.com, high", "check": lambda r: "name" in r and "email" in r and "risk_level" in r and "john@co.com" in r},

    {"dim": "Writing", "name": "CEO Summary", "prompt": "Summarize in exactly 3 bullet points for a CEO: 'Transformer models use self-attention to process sequences in parallel. Unlike RNNs, they don't process tokens sequentially, enabling GPU utilization. The attention mechanism computes relationships between all token pairs. Multi-head attention allows capturing different relationship types. Positional encodings inject order information. Layer normalization and residual connections stabilize deep training. Feed-forward networks apply pointwise transformations. This architecture scales effectively to billions of parameters.'", "check": lambda r: r.count("•") >= 3 or r.count("-") >= 3 or r.count("*") >= 3 or r.count("\n") >= 3},
    {"dim": "Writing", "name": "Delay Email", "prompt": "Write a professional email to a client explaining a 2-week project delay due to unexpected technical debt, proposing a revised timeline and mitigation steps.", "check": lambda r: "delay" in r.lower() and ("technical debt" in r.lower() or "revised" in r.lower() or "timeline" in r.lower())},

    {"dim": "Structured Output", "name": "Entity Extraction", "prompt": 'Extract name, email, company, pain_points as JSON: "Hi, I\'m Sarah Chen (sarah@techcorp.io) from TechCorp. We struggle with slow CI/CD and flaky tests."', "check": lambda r: "sarah" in r.lower() and "techcorp" in r.lower()},
    {"dim": "Structured Output", "name": "NL to SQL", "prompt": "Generate SQL: 'Show all users who signed up in January 2024 and have made at least 3 orders.' Tables: users(id, created_at), orders(id, user_id, created_at)", "check": lambda r: "SELECT" in r.upper() and ("JOIN" in r.upper() or "FROM" in r.upper())},

    {"dim": "Context", "name": "Long Context QA", "prompt": "Context: The project codename is 'Project Falcon'. The budget is $2.4M. The deadline is Q3 2025.\n\nQuestion: What is the project codename and budget? Answer based ONLY on the text above.", "check": lambda r: "falcon" in r.lower() and "2.4" in r},
    {"dim": "Multilingual", "name": "Japanese Translation", "prompt": "Translate to Japanese: 'Our Q3 revenue increased 12% YoY driven by APAC expansion.'", "check": lambda r: len(r) > 3 and any(c in r for c in "増収益四半期割合%12ABCDFGHJKLMNPQRSTUVWXYZabcdefghjklmnpqrstuvwxyzアパック") or len(r) > 10},

    {"dim": "Math", "name": "Probability Word Problem", "prompt": "A bag contains 3 red marbles and 5 blue marbles. Two marbles are drawn without replacement. What is the probability that both are blue? Show fraction steps.", "check": lambda r: "5/14" in r or "10/28" in r or ("5" in r and "14" in r and ("fraction" in r.lower() or "/" in r))},
    {"dim": "Reasoning", "name": "Logical Deduction", "prompt": "Alice, Bob, and Charlie are sitting in a row. Alice always tells the truth. Bob sometimes lies, sometimes tells the truth. Charlie always lies. Person 1 says 'I am Bob'. Person 2 says 'Person 1 is Charlie'. Person 3 says 'Person 2 is Bob'. Who is who? Deduce step by step.", "check": lambda r: "alice" in r.lower() and "bob" in r.lower() and "charlie" in r.lower()},
    {"dim": "Coding", "name": "Two Sum Algorithm", "prompt": "Write an efficient Python function `two_sum(nums, target)` that returns indices of the two numbers such that they add up to target. Aim for O(n) time complexity using a hash map.", "check": lambda r: "def two_sum" in r and ("dict" in r or "hash" in r or "seen" in r or "enumerate" in r)},
    {"dim": "Instruction Following", "name": "Line Start Constraint", "prompt": "Write a 3-line poem about the ocean. Every single line must start strictly with the letter 'B' (case-insensitive).", "check": lambda r: len([line for line in r.strip().split('\n') if line.strip().lower().startswith('b')]) >= 3},

    # --- Newly Added Titles (Robotics, Office Work, AI) ---
    {
        "dim": "Robotics",
        "name": "Robot Navigation Safety",
        "prompt": "An autonomous delivery robot encounters an unexpected obstacle (a small child crossing the path) in a hallway. What are the priority safety layers and sensor inputs it must process immediately to avoid a collision? List 3 key steps.",
        "check": lambda r: "sensor" in r.lower() or "lidar" in r.lower() or "stop" in r.lower() or "emergency" in r.lower() or "brake" in r.lower()
    },
    {
        "dim": "Office Work",
        "name": "Excel VLOOKUP/XLOOKUP",
        "prompt": "Write an Excel formula to look up an employee's department based on their Employee ID in cell A2, searching from a table in Sheet2!A:C where ID is in column A and Department is in column C. Give both VLOOKUP and XLOOKUP options.",
        "check": lambda r: "vlookup" in r.lower() or "xlookup" in r.lower()
    },
    {
        "dim": "AI Concepts",
        "name": "RAG Explanation",
        "prompt": "Explain what RAG (Retrieval-Augmented Generation) means in AI in 2 clear sentences, highlighting why it helps prevent hallucinations.",
        "check": lambda r: ("retrieval" in r.lower() or "retrieve" in r.lower()) and ("hallucination" in r.lower() or "external" in r.lower() or "database" in r.lower())
    }
]

def clean_response(text: str) -> str:
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    return cleaned

results = []
print(f"{'='*60}\nRUNNING {len(TESTS)} TESTS (With Robotics, Office, & AI)\n{'='*60}")

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}/{len(TESTS)}] {test['dim']} - {test['name']}")

    messages = [{"role": "user", "content": test["prompt"]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - start

    input_len = inputs["input_ids"].shape[-1]
    new_toks = outputs.shape[1] - input_len

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    cleaned = clean_response(raw)

    passed = test["check"](cleaned)
    score = 1 if passed else 0
    vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)

    results.append({
        "dimension": test["dim"],
        "test": test["name"],
        "passed": passed,
        "score": score,
        "tok_s": round(new_toks / elapsed, 1),
        "latency_s": round(elapsed, 2),
        "vram_mb": round(vram_mb, 1),
        "response": cleaned[:300] + "..." if len(cleaned) > 300 else cleaned,
    })

    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status} | Speed: {round(new_toks / elapsed, 1)} tok/s | Time: {round(elapsed, 2)}s | VRAM: {round(vram_mb, 1)}MB")

    del inputs, outputs, raw
    torch.cuda.empty_cache()
    gc.collect()

# Summary
print(f"\n{'='*60}\nDIMENSION SUMMARY\n{'='*60}")
dim_scores = {}
for r in results:
    dim_scores.setdefault(r["dimension"], {"pass": 0, "total": 0, "tok_s": [], "latency": []})
    dim_scores[r["dimension"]]["pass"] += r["score"]
    dim_scores[r["dimension"]]["total"] += 1
    dim_scores[r["dimension"]]["tok_s"].append(r["tok_s"])
    dim_scores[r["dimension"]]["latency"].append(r["latency_s"])

summary_rows = []
for dim, stats in dim_scores.items():
    pct = (stats["pass"] / stats["total"]) * 100
    avg_tok = sum(stats["tok_s"]) / len(stats["tok_s"])
    avg_lat = sum(stats["latency"]) / len(stats["latency"])
    summary_rows.append([dim, f"{stats['pass']}/{stats['total']}", f"{pct:.0f}%", f"{avg_tok:.1f}", f"{avg_lat:.2f}s"])

print(tabulate(summary_rows, headers=["Dimension", "Passed", "Score", "Avg tok/s", "Avg latency"], tablefmt="github"))

total_pass = sum(r["score"] for r in results)
total_all = len(results)
print(f"\nOVERALL: {total_pass}/{total_all} ({total_pass/total_all*100:.0f}%)")

RUNNING 19 TESTS (With Robotics, Office, & AI)

[1/19] Reasoning - Train Problem
  ✅ PASS | Speed: 12.3 tok/s | Time: 52.95s | VRAM: 11195.7MB

[2/19] Reasoning - Bat & Ball
  ✅ PASS | Speed: 12.2 tok/s | Time: 33.76s | VRAM: 11195.7MB

[3/19] Coding - LRU Cache
  ❌ FAIL | Speed: 12.3 tok/s | Time: 64.98s | VRAM: 11195.7MB

[4/19] Coding - Bug Fix
  ❌ FAIL | Speed: 12.4 tok/s | Time: 64.57s | VRAM: 11195.7MB

[5/19] Coding - Regex Explain
  ✅ PASS | Speed: 12.3 tok/s | Time: 64.92s | VRAM: 11195.7MB

[6/19] Instruction Following - JSON Only
  ✅ PASS | Speed: 12.3 tok/s | Time: 37.61s | VRAM: 11195.7MB

[7/19] Writing - CEO Summary
  ✅ PASS | Speed: 11.9 tok/s | Time: 29.71s | VRAM: 11195.7MB

[8/19] Writing - Delay Email
  ✅ PASS | Speed: 12.3 tok/s | Time: 58.69s | VRAM: 11195.7MB

[9/19] Structured Output - Entity Extraction
  ✅ PASS | Speed: 12.2 tok/s | Time: 34.55s | VRAM: 11195.7MB

[10/19] Structured Output - NL to SQL
  ✅ PASS | Speed: 12.2 tok/s | Time: 65.54s | VRAM: 11195.7M